In [ ]:
import numpy as np
import pandas as pd
import ast
import time
import tracemalloc
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, RepeatVector, TimeDistributed
from sklearn.metrics import precision_score, recall_score, f1_score
import shap

# --- 1. CONFIG ---
channel = 'P-1' 

# --- 2. LOAD DATA ---
# Loading the files you just unzipped
X_train = np.load(f'data/train/{channel}.npy')
X_test = np.load(f'data/test/{channel}.npy')

# --- 3. PARSE LABELS ---
labels_df = pd.read_csv('labeled_anomalies.csv')
channel_info = labels_df[labels_df['chan_id'] == channel].iloc[0]
anomaly_sequences = ast.literal_eval(channel_info['anomaly_sequences'])

# Create ground truth for the test set (1 = anomaly, 0 = normal)
y_test = np.zeros(len(X_test))
for seq in anomaly_sequences:
    start, end = seq
    y_test[start:end] = 1

In [ ]:
def run_zscore(train_data, test_data, threshold=3.5):
    start_time = time.time()
    tracemalloc.start()
    
    # Use only the first feature (index 0) as per Telemanom README
    train_val = train_data[:, 0]
    test_val = test_data[:, 0]
    
    mean, std = np.mean(train_val), np.std(train_val)
    z_scores = np.abs((test_val - mean) / std)
    predictions = (z_scores > threshold).astype(int)
    
    _, peak_ram = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return predictions, (time.time() - start_time)*1000, peak_ram / (1024*1024)

z_preds, z_time, z_ram = run_zscore(X_train, X_test)

: 

In [ ]:
# Prepare sequences (lookback of 50 timesteps)
def to_sequences(data, seq_size=50):
    x = []
    for i in range(len(data) - seq_size):
        x.append(data[i:(i + seq_size)])
    return np.array(x)

X_train_seq = to_sequences(X_train)
X_test_seq = to_sequences(X_test)
y_test_seq = y_test[50:] # Match sequence length

# Build Autoencoder
model = Sequential([
    LSTM(64, activation='relu', input_shape=(50, X_train.shape[1]), return_sequences=False),
    RepeatVector(50),
    LSTM(64, activation='relu', return_sequences=True),
    TimeDistributed(Dense(X_train.shape[1]))
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train_seq, X_train_seq, epochs=5, batch_size=64, verbose=1)

# Inference
start_time = time.time()
tracemalloc.start()

reconstructed = model.predict(X_test_seq)
mse = np.mean(np.power(X_test_seq - reconstructed, 2), axis=(1,2))
threshold_lstm = np.percentile(mse, 95) # Flag top 5% as anomalies
lstm_preds = (mse > threshold_lstm).astype(int)

_, peak_ram_lstm = tracemalloc.get_traced_memory()
tracemalloc.stop()
lstm_time = (time.time() - start_time)*1000

In [ ]:
# Explain 1 sample using a small background set
explainer = shap.GradientExplainer(model, X_train_seq[:100])
shap_values = explainer.shap_values(X_test_seq[0:1])

print("SHAP values calculated. Feature 0 is the primary telemetry driver.")

In [ ]:
results = pd.DataFrame({
    "Method": ["Z-Score", "LSTM AE"],
    "Precision": [precision_score(y_test, z_preds), precision_score(y_test_seq, lstm_preds)],
    "Recall": [recall_score(y_test, z_preds), recall_score(y_test_seq, lstm_preds)],
    "F1": [f1_score(y_test, z_preds), f1_score(y_test_seq, lstm_preds)],
    "Inf Time (ms)": [z_time/len(X_test), lstm_time/len(X_test_seq)],
    "RAM (MB)": [z_ram, peak_ram_lstm/(1024*1024)]
})
print(results)